# EfficientNetB0

This standalone notebook trains **EfficientNetB0**, evaluates the float model, exports dynamic-range and full INT8 TFLite models, benchmarks CPU inference, and archives outputs.

## Dataset availability

The setup cell below uses the persistent Drive dataset when available. Otherwise, it downloads the prepared Kaggle dataset, validates its splits, and saves it to Drive for future runtimes.


In [ ]:
# The Drive-backed dataset setup cell below downloads Kaggle data only when the persistent Drive copy is unavailable.


## Clone Project Repository

In [ ]:
from google.colab import drive
import os
import subprocess

drive.mount("/content/drive")

ARCH = "efficientnet_b0"
REPO_URL = "https://github.com/hit1363/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System.git"
REPO_DIR = "/content/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System"
OUTPUT_BASE = f"/content/drive/MyDrive/leaf_models/{ARCH}"
print("Configured:", ARCH)
print("Outputs:", OUTPUT_BASE)


if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    sync_result = subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", "main"], text=True, capture_output=True)
    if sync_result.returncode == 0:
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
    print(sync_result.stdout, sync_result.stderr)
required_paths = ["requirements.txt", "training/train.py", "training/evaluate.py", "quantization/post_training_quant.py"]
missing_paths = [path for path in required_paths if not os.path.isfile(os.path.join(REPO_DIR, path))]
if missing_paths:
    raise RuntimeError(f"Repository clone is incomplete; missing: {missing_paths}")
print("Complete project repository is ready:", REPO_DIR)

## Install project dependencies

In [ ]:
import os
import subprocess
import sys

requirements_path = os.path.join(REPO_DIR, "requirements.txt")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path], check=True)
print("Dependencies installed from:", requirements_path)

## Configure paths and detect classes

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

from google.colab import drive, files  # pyright: ignore[reportMissingImports]

KAGGLE_DATASET_ID = "mdhasibultamim/multi-crop-leaf-disease"
DRIVE_DATASET = Path("/content/drive/MyDrive/leaf_data/processed")
LOCAL_DATASET = Path("/content/leaf_data/processed")
LOCAL_ROOT = LOCAL_DATASET.parent
IMAGE_SUFFIXES = {".bmp", ".gif", ".jpeg", ".jpg", ".png"}

drive.mount("/content/drive")

def inspect_dataset(dataset_dir):
    """Return the shared class names when all split folders are usable."""
    def has_image(root_dir):
        for _, _, filenames in os.walk(root_dir):
            if any(Path(filename).suffix.lower() in IMAGE_SUFFIXES for filename in filenames):
                return True
        return False

    split_classes = {}
    for split in ("train", "val", "test"):
        split_path = dataset_dir / split
        if not split_path.is_dir():
            return None
        classes = sorted(child.name for child in split_path.iterdir() if child.is_dir())
        if not classes or not has_image(split_path):
            return None
        split_classes[split] = classes

    expected = split_classes["train"]
    if any(classes != expected for classes in split_classes.values()):
        raise ValueError(
            "Dataset split class folders differ: "
            + "; ".join(f"{split}={len(classes)}" for split, classes in split_classes.items())
        )
    for split, classes in split_classes.items():
        print(f"  ✓ {split}: {len(classes)} classes")
    return expected

print("Checking Drive dataset:", DRIVE_DATASET)
drive_class_names = inspect_dataset(DRIVE_DATASET) if DRIVE_DATASET.exists() else None

if drive_class_names is not None:
    print("Found Drive dataset at:", DRIVE_DATASET)
    if LOCAL_DATASET.exists():
        shutil.rmtree(LOCAL_DATASET)
    LOCAL_DATASET.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_DATASET, LOCAL_DATASET)
    print("Copied Drive dataset to local runtime:", LOCAL_DATASET)
else:
    if DRIVE_DATASET.exists():
        raise RuntimeError(
            f"Drive dataset is incomplete: {DRIVE_DATASET}. Expected non-empty train, val, and test splits."
        )
    print("Drive dataset not found; downloading the Kaggle dataset into /content.")
    download_dir = LOCAL_ROOT / "kaggle_download"
    extract_dir = LOCAL_ROOT / "kaggle_extracted"
    kaggle_config_dir = Path.home() / ".kaggle"
    kaggle_config = kaggle_config_dir / "kaggle.json"
    download_dir.mkdir(parents=True, exist_ok=True)
    extract_dir.mkdir(parents=True, exist_ok=True)

    if not kaggle_config.is_file():
        print("Upload kaggle.json from Kaggle Account → Create New API Token.")
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise FileNotFoundError("Please upload a file named kaggle.json")
        kaggle_config_dir.mkdir(parents=True, exist_ok=True)
        kaggle_config.write_bytes(uploaded["kaggle.json"])
    os.chmod(kaggle_config, 0o600)
    os.environ["KAGGLE_CONFIG_DIR"] = str(kaggle_config_dir)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", KAGGLE_DATASET_ID, "-p", str(download_dir), "--force"],
        check=True,
    )
    archives = sorted(download_dir.glob("*.zip"), key=lambda path: path.stat().st_mtime)
    if not archives:
        raise FileNotFoundError(f"No Kaggle archive was downloaded into {download_dir}")
    with zipfile.ZipFile(archives[-1]) as archive:
        archive.extractall(extract_dir)

    candidates = [extract_dir] + [path for path in extract_dir.rglob("processed") if path.is_dir()]
    valid_candidates = [(path, inspect_dataset(path)) for path in candidates]
    valid_candidates = [(path, classes) for path, classes in valid_candidates if classes is not None]
    if len(valid_candidates) != 1:
        raise RuntimeError(
            "Could not find exactly one prepared train/val/test dataset in the Kaggle archive. "
            f"Found {len(valid_candidates)} valid candidates."
        )
    source_dataset, _ = valid_candidates[0]
    if LOCAL_DATASET.exists():
        shutil.rmtree(LOCAL_DATASET)
    shutil.copytree(source_dataset, LOCAL_DATASET)
    DRIVE_DATASET.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(LOCAL_DATASET, DRIVE_DATASET)
    print("Saved prepared dataset to Drive for future runtimes:", DRIVE_DATASET)

class_names = inspect_dataset(LOCAL_DATASET)
if class_names is None:
    raise RuntimeError(f"Local dataset setup failed: {LOCAL_DATASET}")
DATASET_BASE = str(LOCAL_DATASET)
print("Training/evaluation dataset:", DATASET_BASE)
print("Classes:", len(class_names))

import yaml

cfg_path = os.path.join(REPO_DIR, "training", f"config_{ARCH}.yaml")
with open(cfg_path, "r", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)

if ARCH == "efficientnet_lite0":
    cfg["model"]["hub_url"] = "https://tfhub.dev/tensorflow/efficientnet/lite0/feature-vector/2"
    tfhub_cache_dir = "/content/tfhub_modules"
    os.makedirs(tfhub_cache_dir, exist_ok=True)
    os.environ["TFHUB_CACHE_DIR"] = tfhub_cache_dir
    cfg["model"]["hub_cache_dir"] = tfhub_cache_dir

cfg["dataset"]["data_dir"] = DATASET_BASE
for split in ("train", "val", "test"):
    cfg["dataset"][f"{split}_dir"] = os.path.join(DATASET_BASE, split)
cfg["model"]["num_classes"] = len(class_names)
cfg["training"]["epochs"] = 25
cfg["training"]["freeze_base"] = True
cfg["training"]["unfreeze_epoch"] = 5
cfg["training"]["unfreeze_from_layer"] = 50
cfg["training"]["fine_tune_learning_rate"] = 0.0001
cfg["lr_schedule"].update({
    "type": "reduce_on_plateau",
    "monitor": "val_loss",
    "factor": 0.5,
    "patience": 5,
    "min_lr": 1e-7,
})
cfg["callbacks"]["early_stopping"]["patience"] = 5
cfg["callbacks"]["tensorboard"]["enabled"] = False
cfg["dataset"]["cache_mode"] = "disk"
cfg["dataset"]["cache_memory_limit_gb"] = 8.0
cfg.setdefault("performance", {}).update({
    "mixed_precision": True,
    "steps_per_execution": 32,
    "jit_compile": False,
    "deterministic": False,
    "distribution": "auto",
    "log_throughput": True,
})
cfg["training"].update({
    "epochs": 25,
    "freeze_base": True,
    "unfreeze_epoch": 5,
    "unfreeze_from_layer": 50,
    "fine_tune_learning_rate": 0.0001,
})
cfg["lr_schedule"].update({
    "type": "reduce_on_plateau",
    "monitor": "val_loss",
    "factor": 0.5,
    "patience": 5,
    "min_lr": 1e-7,
})
cfg["callbacks"]["early_stopping"]["patience"] = 5
cfg["callbacks"]["tensorboard"]["enabled"] = False
cfg["export"]["save_dir"] = os.path.dirname(OUTPUT_BASE)
cfg["callbacks"]["csv_logger"]["filename"] = os.path.join(OUTPUT_BASE, f"training_log_{ARCH}.csv")
os.makedirs(OUTPUT_BASE, exist_ok=True)
with open(cfg_path, "w", encoding="utf-8") as handle:
    yaml.safe_dump(cfg, handle, sort_keys=False)

print(f"{ARCH}: {len(class_names)} classes")
print("Config:", cfg_path)


## Train the model

In [ ]:
train_result = subprocess.run([
    sys.executable,
    os.path.join(REPO_DIR, "training", "train.py"),
    "--config",
    cfg_path,
], text=True, capture_output=True)
print(train_result.stdout)
print(train_result.stderr)
if train_result.returncode != 0:
    raise RuntimeError(f"Training failed with exit code {train_result.returncode}")

## Evaluate the float model

In [ ]:
from pathlib import Path

arch_dir = Path(OUTPUT_BASE)
saved_models = sorted(arch_dir.glob("saved_model_*"), key=lambda path: path.stat().st_mtime)
h5_models = sorted(arch_dir.glob("*.h5"), key=lambda path: path.stat().st_mtime)
if saved_models:
    model_path = str(saved_models[-1])
elif h5_models:
    model_path = str(h5_models[-1])
else:
    raise FileNotFoundError(f"No exported model found in {OUTPUT_BASE}")

float_report = os.path.join(OUTPUT_BASE, "evaluation_float")
os.makedirs(float_report, exist_ok=True)
subprocess.run([
    sys.executable,
    os.path.join(REPO_DIR, "training", "evaluate.py"),
    "--model", model_path,
    "--config", cfg_path,
    "--results-dir", float_report,
], check=True)
print("Evaluated:", model_path)

## Export dynamic-range and full INT8 TFLite models

In [ ]:
quant_script = os.path.join(REPO_DIR, "quantization", "post_training_quant.py")
quant_dir = os.path.join(OUTPUT_BASE, "quantized")
os.makedirs(quant_dir, exist_ok=True)

dynamic_path = os.path.join(quant_dir, f"{ARCH}_dynamic.tflite")
subprocess.run([
    sys.executable, quant_script,
    "--model_path", model_path,
    "--output_path", dynamic_path,
    "--arch", ARCH,
], check=True)

int8_path = os.path.join(quant_dir, f"{ARCH}_int8.tflite")
subprocess.run([
    sys.executable, quant_script,
    "--model_path", model_path,
    "--output_path", int8_path,
    "--representative_data", cfg["dataset"]["train_dir"],
    "--evaluate",
    "--test_data", cfg["dataset"]["test_dir"],
    "--arch", ARCH,
], check=True)
print("Exported:", dynamic_path, int8_path)

## Benchmark and evaluate TFLite models

In [ ]:
import json
import time
import numpy as np
import tensorflow as tf

benchmark_rows = []
for tflite_path in (dynamic_path, int8_path):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    input_shape = input_details["shape"]
    if input_details["dtype"] == np.uint8:
        dummy = np.random.randint(0, 256, size=input_shape, dtype=np.uint8)
    else:
        dummy = np.random.uniform(-1.0, 1.0, size=input_shape).astype(np.float32)
    for _ in range(10):
        interpreter.set_tensor(input_details["index"], dummy)
        interpreter.invoke()
    times = []
    for _ in range(50):
        start = time.perf_counter()
        interpreter.set_tensor(input_details["index"], dummy)
        interpreter.invoke()
        times.append((time.perf_counter() - start) * 1000.0)
    row = {
        "model": os.path.basename(tflite_path),
        "input_dtype": np.dtype(input_details["dtype"]).name,
        "mean_ms": float(np.mean(times)),
        "std_ms": float(np.std(times)),
        "size_mb": os.path.getsize(tflite_path) / (1024 * 1024),
    }
    benchmark_rows.append(row)
    print(json.dumps(row, indent=2))

with open(os.path.join(OUTPUT_BASE, "tflite_benchmark.json"), "w", encoding="utf-8") as handle:
    json.dump(benchmark_rows, handle, indent=2)

## Archive outputs

In [ ]:
import shutil

archive_base = os.path.join(os.path.dirname(OUTPUT_BASE), f"{ARCH}_outputs")
archive_path = shutil.make_archive(archive_base, "zip", OUTPUT_BASE)
print("Archive:", archive_path)

## Outputs

The model, class labels, logs, float evaluation report, quantized TFLite files, benchmark JSON, and ZIP archive are stored under the configured output directory.